# Merge pairs - pRESTO AssemblePairs

One-Q notebook for PRJEB40348. Input: `/data/user/epishkin/results/PRJEB40348/trimmed/fastq`. Output: `/data/user/epishkin/results/PRJEB40348/merged/{fastq,logs,qc}`. This step only merges already trimmed paired reads


In [ ]:
import os, sys, sysconfig
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
# Remove persistent user-site packages so pRESTO resolves from bcr_env, not ~/.local.
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)


In [ ]:
import gzip, shutil, subprocess
from pathlib import Path

VOLUME = Path("/data/user/epishkin")
DATASET = "PRJEB40348"
INPUT_DIR = VOLUME / "results" / DATASET / "trimmed" / "fastq"
OUT_BASE = VOLUME / "results" / DATASET / "merged"
OUT_FASTQ = OUT_BASE / "fastq"
OUT_LOGS = OUT_BASE / "logs"
OUT_QC = OUT_BASE / "qc"
for d in (OUT_FASTQ, OUT_LOGS, OUT_QC):
    d.mkdir(parents=True, exist_ok=True)


## 1. Check pRESTO

Run this in the BCR Pipeline kernel. If pRESTO is missing, run the install cell below.


In [ ]:
def check_presto():
    assemble = shutil.which("AssemblePairs.py")
    parse_log = shutil.which("ParseLog.py")
    try:
        import presto
        module_status = "OK"
    except Exception as exc:
        module_status = "NOT FOUND: " + repr(exc)
    print(f"AssemblePairs.py: {assemble or 'NOT FOUND'}")
    print(f"ParseLog.py:      {parse_log or 'NOT FOUND'}")
    print(f"presto module:    {module_status}")
    return bool(assemble)
presto_ok = check_presto()


In [ ]:
# Run only if pRESTO is missing in the BCR Pipeline kernel.
# subprocess.run([sys.executable, "-m", "pip", "install", "presto"], check=True)
# presto_ok = check_presto()


## 2. Discover read pairs

Supports both `*.pr.fastq.gz` and `*.trim.fastq.gz` outputs.


In [ ]:
def discover_pairs(input_dir):
    patterns = [
        ("*_1.pr.fastq.gz", "_1.pr.fastq.gz", "_2.pr.fastq.gz"),
        ("*_1.trim.fastq.gz", "_1.trim.fastq.gz", "_2.trim.fastq.gz"),
    ]
    pairs, seen = [], set()
    for glob_pat, r1_suffix, r2_suffix in patterns:
        for r1 in sorted(input_dir.glob(glob_pat)):
            sample = r1.name[:-len(r1_suffix)]
            if sample in seen:
                continue
            r2 = input_dir / f"{sample}{r2_suffix}"
            if r2.exists():
                pairs.append((sample, r1, r2)); seen.add(sample)
            else:
                print(f"WARNING: missing mate for {r1.name}")
    return pairs
pairs = discover_pairs(INPUT_DIR)
print(f"Input directory: {INPUT_DIR}")
print(f"Found pairs: {len(pairs)}")
pairs[:3]


## 3. AssemblePairs

Uses `AssemblePairs.py align --coord illumina --rc tail --failed`.


In [ ]:
def run_assemble_pairs(sample, r1, r2, force=False):
    pass_fastq = OUT_FASTQ / f"{sample}_assemble-pass.fastq"
    fail_fastq = OUT_FASTQ / f"{sample}_assemble-fail.fastq"
    log_path = OUT_LOGS / f"{sample}.assemble.log"
    stdout_path = OUT_LOGS / f"{sample}.stdout.txt"
    stderr_path = OUT_LOGS / f"{sample}.stderr.txt"
    if pass_fastq.exists() and not force:
        print(f"[skip] {sample}: {pass_fastq.name} exists")
        return sample, "skipped", pass_fastq, fail_fastq, log_path
    cmd = ["AssemblePairs.py", "align", "-1", str(r1), "-2", str(r2), "--coord", "illumina", "--rc", "tail", "--outname", sample, "--outdir", str(OUT_FASTQ), "--log", str(log_path), "--failed"]
    print("[run]", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    stdout_path.write_text(res.stdout); stderr_path.write_text(res.stderr)
    if res.returncode != 0:
        raise RuntimeError(f"AssemblePairs failed for {sample}; see {stderr_path}")
    return sample, "done", pass_fastq, fail_fastq, log_path


In [ ]:
# Smoke test on first run.
if not pairs:
    raise FileNotFoundError(f"No pairs found in {INPUT_DIR}")
if not shutil.which("AssemblePairs.py"):
    raise RuntimeError("AssemblePairs.py not found; run install cell or fix BCR Pipeline environment.")
smoke_result = run_assemble_pairs(*pairs[0], force=False)
smoke_result


In [ ]:
# Full run across all PRJEB40348 pairs.
results = [run_assemble_pairs(sample, r1, r2, force=False) for sample, r1, r2 in pairs]
manifest_path = OUT_QC / "assemble_manifest.tsv"
with open(manifest_path, "w") as out:
    out.write("sample	status	pass_fastq	fail_fastq	log
")
    for sample, status, pass_fastq, fail_fastq, log_path in results:
        out.write(f"{sample}	{status}	{pass_fastq}	{fail_fastq}	{log_path}
")
print(f"Wrote {manifest_path}")


## 4. Merge QC summary

Counts pass/fail FASTQ records and writes `assembly_qc.tsv`.


In [ ]:
def count_fastq_records(path):
    path = Path(path)
    if not path.exists():
        return 0
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt") as handle:
        return sum(1 for _ in handle) // 4
qc_path = OUT_QC / "assembly_qc.tsv"
total_pass = total_fail = 0
with open(qc_path, "w") as out:
    out.write("sample	assembled_pairs	failed_pairs	total_pairs_seen	merge_rate	pass_fastq	fail_fastq	log
")
    for sample, status, pass_fastq, fail_fastq, log_path in results:
        n_pass, n_fail = count_fastq_records(pass_fastq), count_fastq_records(fail_fastq)
        total = n_pass + n_fail; rate = n_pass / total if total else 0
        total_pass += n_pass; total_fail += n_fail
        out.write(f"{sample}	{n_pass}	{n_fail}	{total}	{rate:.6f}	{pass_fastq}	{fail_fastq}	{log_path}
")
print(f"Wrote {qc_path}")
print("Total assembled pairs:", total_pass)
print("Total failed pairs:   ", total_fail)
print("Overall merge rate:  ", total_pass / (total_pass + total_fail) if (total_pass + total_fail) else 0)


## 5. Optional FastQC + MultiQC

Run after full merge if `fastqc` and `multiqc` are available.


In [ ]:
def run_qc_merged():
    pass_files = sorted(OUT_FASTQ.glob("*_assemble-pass.fastq"))
    if not pass_files:
        raise FileNotFoundError(f"No assembled pass FASTQ files in {OUT_FASTQ}")
    fastqc_dir = OUT_QC / "fastqc"; multiqc_dir = OUT_QC / "multiqc"
    fastqc_dir.mkdir(parents=True, exist_ok=True); multiqc_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["fastqc", "-t", "4", "-q", "--noextract", *map(str, pass_files), "-o", str(fastqc_dir)], check=True)
    subprocess.run(["multiqc", str(fastqc_dir), str(OUT_LOGS), "-o", str(multiqc_dir), "-n", f"{DATASET}_merged_multiqc", "-f"], check=True)
    return multiqc_dir / f"{DATASET}_merged_multiqc.html"
# merged_multiqc = run_qc_merged()
# merged_multiqc
